# Tokenizacja

W trakcie dzisiejszych zajęć poznamy głębiej problem tokenizacji, czyli segmentacji tekstów na tokeny. Tokeny to jednostki zgrubsza odpowiadające słowom, i.e. jednostki pomiędzy najniższym poziomem analizy (znakami), a wyższymi poziomami (np. paragrafami, chunkami, plikami).

Tokenizacja jest często pierwszym etapem pracy z tekstem. Jej znaczenie jest fundamentalne, ponieważ wszelkie niedociągnięcia jakie popełnimy na tym etapie będą propagowane w kolejnych elementach potoku przetwarzania tekstu. To na jakie segmenty podzielimy tekst, wyznacza więc granice możliwości naszej analizy. Nie zawsze decyzja ta będzie podlegała bezsprzecznym kryteriom - czasami charakterystyka danych będzie wymuszała decyzje, które w innej domenie byłyby szkodliwe.

In [ ]:
import os
import pandas
import re
from pathlib import Path

Skorzystamy z danych pochodzących z serwisu https://wolnelektury.pl/

Pobraliśmy stamtąd zbiór klasycznych dramatów greckich, wpierw wykorzystajmy znaną nam metodę eksploracji katalogów, aby zebrać ścieżki do interesujących nas plików:

In [ ]:
DATA_DIR = Path("../../data")
books_dir = DATA_DIR / "lektury" / "dramaty"
filepaths = []
for dir_path, subdirs, files in os.walk(books_dir):
    for file in files:
        if file.endswith(".txt"):
            filepath = Path(dir_path) / file
            print(filepath)
            filepaths.append(filepath)


Następnie wczytajmy do tabeli treść plików, i podzielmy zgrubnie każdy z tekstów korzystając z metody `split`:

In [ ]:
def read_file(fpath): # helper function
    with open(fpath) as f:
        txt = f.read()
    return txt

df = pandas.DataFrame({"fpath": filepaths})
df["text"] = df["fpath"].apply(read_file)
df["pretokens"] = df["text"].apply(lambda x: x.split())

first_file = df.iloc[0]
print(first_file["text"][:200])
print(first_file["pretokens"][:50])

## Ćwiczenie 1:

Korzystając ze znanych już narzędzi biblioteki `pandas` przygotuj sumaryczną listę frekwencyjną segmentów (wyodrębnionych w kolumnie "pretokens") występujących w korpusie

In [ ]:
# TODO

Wykorzystajmy naszą wiedzę o regexach do konstrukcji prostego tokenizatora. Podamy mu trzy typy ciągów znaków które chcemy traktować jako tokeny:

In [ ]:
simple_tokenizer = re.compile(r"\b\w+\b|"   #sequences of letters at word boundaries
                              r"\d+|"       #sequences of digits
                              r"[.,!?;—…*-]"#punctuation
                             )

df["simple_tokens"] = df["text"].apply(lambda x: [m[0] # [0] on Match  object extracts the entire matching string
                                                  for m in simple_tokenizer.finditer(x)])

Zobaczmy jak różnią się wyniki takiej tokenizacji, od wbudowanej w Pythona metody `split`:

In [ ]:
first_file = df.iloc[0]
pretokens, simple_tokens = first_file["pretokens"], first_file["simple_tokens"]
for i, (ptok, stok) in enumerate(list(zip(pretokens, simple_tokens))[:100]):
    print(f"{i}\t{ptok}\t{stok}")

## Ćwiczenie 2:

W tekście znajdują się cztery URLe. Zmodyfikuj wyrażenie regularne wykorzystane do tokenizacji tak, aby potraktował te elementy jako pojedyncze, nierozbijalne tokeny

In [ ]:
# TODO

## Ćwiczenie 3:

Spróbuj zrekonstruować oryginalny tekst na podstawie tokenów, które ze zbadanych dotychczas tokenizatorów na to pozwalają?

In [ ]:
# TODO

## Tokenizacja BPE
W pewnych zastosowaniach istotne będzie skorzystanie z tokenizacji która rozkłada wyrazy "od spacji do spacji" na mniejsze jednostki, takie rodzaje tokenizatorow określamy często mianem "sub-word". W takim podejściu moglibyśmy np. rozbijać słowo "niejasny" na dwa tokeny: "nie" oraz "jasny" (czasami będziemy też chcieli zapamiętać informację o tym, że "nie" stanowi prefiks, w odróżnieniu od samodzielnego słowa - moglibyśmy to zasygnalizować np. zapisując "nie-")
Głównymi powodami dla którego chcielibyśmy to zrobić są:
1. ekonomia - redukcja liczności zbioru tokenów z którymi mamy do czynienia
1. wykorzystanie informacji zawartej w strukturze morfologicznej słów

Aby zobrazować tę drugą cechę spójrzmy na słowo:

*jupyterowość*

Nie występuje ono w żadnym słowniku, a jednak intuicyjnie jesteśmy w stanie powiedzieć na czym może polegać jego znaczenie. Wiedzę tę czerpiemy z morfologii tego słowa, tj. rozkładając je na dwie podjednostki: "jupyter" oraz "-owość". Każda z nich jest nam indywidualnie znajoma i dlatego, w oparciu o kompozycyjność, jesteśmy w stanie zrekonstruować zamierzone znaczenie.



Skorzystamy z dwóch dodatkowych bibliotek pomocniczych:

1. `tqdm` pozwala na opakowanie dowolnego iteratora w sposób, który sprawi że iterując przez jego elementy, wyświetli się nam pasek postępu - przydatne przy wymagających obliczeniowo zadaniach.
2. `numpy` jest biblioteką służącą do pracy z macierzami (obiekt `Array`) i algebrą liniową, wykorzystamy ją dzisiaj w podstawowym zakresie.

In [ ]:
import numpy
from tqdm import tqdm

a = numpy.zeros((3,3))
print(type(a))

for row in tqdm(a):
    print(row)

Współczesne tokenizatory sub-word często opierają się na metodzie *Byte Pair Encoding* (**BPE**). Tokenizator taki musi zostać wytrenowany na zbiorze danych (korpusie), poprzez zastosowanie prostego algorytmu:

1. Zidentyfikuj zbiór **znaków** występujących w korpusie - będzie to początkowy zbiór tokenów
1. Przetokenizuj korpus
1. Iteratywnie:
    1. Oblicz statystyki następstwa po sobie (sukcesji) tokenów
    2. Wybierz najczęściej występującą parę tokenów
    3. dodaj do zbioru tokenów połączenie w.w. pary
    4. ponownie przetokenizuj korpus

In [ ]:
char_set = sorted(set(df["text"].sum()))
token_set = [char for char in char_set] # copying
tok2ind = {c:i for i,c in enumerate(token_set)} # tokens will be mapped to indices of array
tokenizer_string = r"." # matches on every individual char

In [ ]:
pretoken_counts = df["pretokens"].explode().apply(lambda x: x.strip()).value_counts()

for merge_num in tqdm(range(1000)):
    bpe_tokenizer = re.compile(tokenizer_string) # recreate tokenizer with new merges
    num_toks = len(token_set)
    succ_count = numpy.zeros((num_toks, num_toks)) # initialize succession array succ_count[j][k] = number of successions of j by k
    for pretok, freq in pretoken_counts.items():
        segmented = bpe_tokenizer.findall(pretok) # retokenize
        for i in range(len(segmented) - 1):
            j = tok2ind[segmented[i]]
            k = tok2ind[segmented[i+1]]
            succ_count[j][k] += freq # saving statistics

    most_freq = succ_count.argmax() # select top merge and decompose the flat index into j and k
    j = most_freq // num_toks
    k = most_freq % num_toks
    j_tok = token_set[j]
    k_tok = token_set[k]
    token_set.append(j_tok + k_tok) # add the newly merged token
    if succ_count[j][k] == succ_count[j].sum():# j_tok is ALWAYS followed by k_tok
        token_set.remove(j_tok) 
    if succ_count[j][k] == succ_count.T[k].sum():# k_tok is ALWAYS preceded by j_tok
        token_set.remove(k_tok)
    tok2ind = {c:i for i,c in enumerate(token_set)}
    tokenizer_string = "|".join([re.escape(tok) for tok in sorted(token_set, key=lambda x: len(x), reverse=True)])
    # handling greediness - because of cases like "ie", "nie" and "ni" introduced in order -> "ni" will have preference over "nie"

Zobaczmy jak sprawuje się wytrenowany tokenizator:

In [ ]:
print(bpe_tokenizer.findall("jupyterowość"))
print(bpe_tokenizer.findall("niespodziewanie"))
df["bpe_tokens"] = df["text"].apply(bpe_tokenizer.findall)

I porównajmy go z poprzednimi:

In [ ]:
first_file = df.iloc[0]
pretokens, simple_tokens, bpe_tokens = first_file["pretokens"], first_file["simple_tokens"], first_file["bpe_tokens"]
for i, (ptok, stok, btok) in enumerate(list(zip(pretokens, simple_tokens, bpe_tokens))[:100]):
    print(f"{i}\t{ptok}\t{stok}\t{btok}")

Spójrzmy na to jakie faktycznie tokeny zostały wyuczone:

In [ ]:
first_merge = len(char_set)
for i, t in enumerate(token_set[first_merge:first_merge+50]):
    print(i+1, t)

print("...")
first_merge = len(char_set)
for i, t in enumerate(token_set[len(token_set)-50:]):
    print(len(token_set) - first_merge - 49 + i, t)

Kilka obserwacji:
1. Pierwsze tokeny odzwierciedlają tak naprawdę specyfikę fonologiczną języka polskiego ("sz", "rz" etc.)
2. bardzo wcześnie pojawiają się tokeny o charakterze morfologicznym ("nie", "wy", "by")
3. później obserwujemy tokeny odpowiadające pełnym słowom ("musi", "równie", "przecież"), niektore z nich odzwierciedlają specyfikę korpusu ("PERIPLEKTOMENUS", "bóg", "Któż").

Finalnie spójrzmy na dojrzały, współczesny tokenizator wytrenowany algorytmem opartym o BPE na dużym zbiorze danych tekstowych. Jest on sparowany z modelem transformerowym o nazwie `allegro/herbert-base-cased` - polskim odpowiednikiem BERTa

https://huggingface.co/allegro/herbert-base-cased

In [ ]:
from transformers import AutoTokenizer

trf_tokenizer = AutoTokenizer.from_pretrained("allegro/herbert-base-cased")
text_sample = first_file["text"][:1000]
trf_tokenized = trf_tokenizer(text_sample)
print(trf_tokenized)

Jak widzimy jego output jest nieco bardziej złożony, i od razu dostrzegamy, że tokeny są kodowane jako liczby porządkowe. Spróbujmy zgłębić jak to kodowanie faktycznie wygląda:

In [ ]:
inv_vocab = {ind: tok for tok, ind in trf_tokenizer.vocab.items()}
for i in range(200):
    print(f"{i} -> {inv_vocab[i]}")

for i in range(2000, 2200):
    print(f"{i} -> {inv_vocab[i]}")

for i in range(20000, 20200):
    print(f"{i} -> {inv_vocab[i]}")

Zobaczmy również to kodowanie na naszym przykładzie:

In [ ]:
print(trf_tokenizer.decode(trf_tokenized["input_ids"]))
for i in trf_tokenized["input_ids"]:
    print(f"{i} -> {inv_vocab[i]}")

Ta metoda tokenizacji wiąże się z co najmniej dwoma mankamentami:

Po pierwsze: Rzadko będziemy w stanie odtworzyć oryginalny tekst z jego stokenizowanej postaci

In [ ]:
print(text_sample)
print("\n=========== vs ============\n")
print(trf_tokenizer.decode(trf_tokenized, skip_special_tokens=True))

Po drugie: nie każda sekwencja tokenów będzie sparowana z tekstem, który wygeneruje ją z powrotem (co jest konsekwencją faktu, że tokenizacja nie jest procedurą odwracalną).

In [ ]:
token_ids = [2000, 3000]
print([inv_vocab[i] for i in token_ids])
detokenized = trf_tokenizer.decode(token_ids)
print(detokenized)
retokenized = trf_tokenizer(detokenized, add_special_tokens=False)["input_ids"]
print(retokenized)
assert retokenized != token_ids

## Ćwiczenie 4
Dla każdego z tokenizatorów, i każdego z tekstów policz:
1. ile tokenów wyodrębniono z tekstu korzystając z tokenizatora
1. ile unikalnych tokenów występuje w tekście. 

Wyniki przedstaw jako tabelkę korzystając z klasy `DataFrame`

In [ ]:
# TODO